In [0]:
"""
Modelo baseline para prever se um pedido vai atrasar.

Usa só o que já se sabe no momento do pedido e separa treino e teste por
período, porque a ideia é prever pedido novo, não acertar o passado.

Requisitos: pip install pandas scikit-learn joblib
"""

import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

CAMINHO = Path('../data/DataCoSupplyChainDataset.csv')
PASTA_SAIDA = Path('output')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

colunas = {
    'Delivery Status': 'status',
    'order date (DateOrders)': 'data_pedido',
    'Shipping Mode': 'modo',
    'Days for shipment (scheduled)': 'dias_prog',
    'Order Region': 'regiao',
    'Market': 'mercado',
    'Category Name': 'categoria',
    'Type': 'tipo_pagamento',
    'Customer Segment': 'segmento',
    'Order Item Quantity': 'quantidade',
}
df = pd.read_csv(CAMINHO, encoding='latin1', usecols=list(colunas)).rename(columns=colunas)

# Pedidos cancelados nunca foram enviados: ficam fora
df = df[df['status'] != 'Shipping canceled'].copy()

# Alvo: 1 = atrasou, 0 = não atrasou
df['atrasado'] = (df['status'] == 'Late delivery').astype(int)

# Variáveis de data (conhecidas no momento do pedido)
df['data_pedido'] = pd.to_datetime(df['data_pedido'], format='%m/%d/%Y %H:%M')
df['mes'] = df['data_pedido'].dt.month
df['dia_semana'] = df['data_pedido'].dt.dayofweek

# Ficam de fora 'Days for shipping (real)', 'Delivery Status' e
# 'Late_delivery_risk': essas colunas só existem depois da entrega, então o
# modelo estaria olhando a resposta.
cat_cols = ['modo', 'regiao', 'mercado', 'categoria', 'tipo_pagamento', 'segmento']
num_cols = ['dias_prog', 'quantidade', 'mes', 'dia_semana']

# Treino com os pedidos mais antigos, teste com os mais recentes
df = df.sort_values('data_pedido')
corte = df['data_pedido'].quantile(0.80)           # 80% mais antigos = treino
treino = df[df['data_pedido'] <= corte]
teste = df[df['data_pedido'] > corte]

print(f"Treino: {len(treino):,} pedidos ({treino['data_pedido'].min().date()} a {treino['data_pedido'].max().date()})")
print(f"Teste:  {len(teste):,} pedidos ({teste['data_pedido'].min().date()} a {teste['data_pedido'].max().date()})")

X_treino, y_treino = treino[cat_cols + num_cols], treino['atrasado']
X_teste, y_teste = teste[cat_cols + num_cols], teste['atrasado']

preparo = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)],
    remainder='passthrough',   # as colunas numéricas passam direto
)
modelo = Pipeline([
    ('preparo', preparo),
    ('floresta', RandomForestClassifier(
        n_estimators=100, min_samples_leaf=50, random_state=42, n_jobs=-1)),
])
modelo.fit(X_treino, y_treino)

# Avalia no período de teste
previsto = modelo.predict(X_teste)

metricas = {
    'pedidos_treino': int(len(treino)),
    'pedidos_teste': int(len(teste)),
    'taxa_atraso_teste_pct': round(float(y_teste.mean() * 100), 1),
    'acuracia': round(accuracy_score(y_teste, previsto), 3),
    'precisao': round(precision_score(y_teste, previsto), 3),
    'recall': round(recall_score(y_teste, previsto), 3),
    'f1': round(f1_score(y_teste, previsto), 3),
}
tn, fp, fn, tp = confusion_matrix(y_teste, previsto).ravel()
metricas.update({'verdadeiros_positivos': int(tp), 'falsos_positivos': int(fp),
                 'falsos_negativos': int(fn), 'verdadeiros_negativos': int(tn)})

metricas['referencia_sempre_atrasa'] = {
    'acuracia': round(float(y_teste.mean()), 3),
    'precisao': round(float(y_teste.mean()), 3),
    'recall': 1.0,
}
print('\n=== MÉTRICAS (teste) ===')
for k, v in metricas.items():
    print(f'{k}: {v}')

# Quais variáveis mais pesaram
nomes = modelo.named_steps['preparo'].get_feature_names_out()
imp = (pd.Series(modelo.named_steps['floresta'].feature_importances_, index=nomes)
       .sort_values(ascending=False).head(8).round(3))
print('\n=== VARIÁVEIS MAIS IMPORTANTES ===')
print(imp.to_string())

joblib.dump(modelo, 'modelo.joblib')
(PASTA_SAIDA / 'metricas.json').write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding='utf-8')
print("\nSalvo: modelo.joblib e output/metricas.json")